# Step 2 — Cross-patient clustering with Seurat: 11 ST clusters on TCGA

Takes the per-domain *averaged* expression profiles produced upstream (one row per `(slide, SpaGCN domain)` × one column per gene) and jointly clusters them across all TCGA-BRCA patients with the Seurat pipeline. Yields **11 ST clusters** consistent across patients.

Adapted from `spatial_clusters/wining_clustering_v2.r` — the "winning" parameter set chosen after a wider grid search (`0_param_grid.r` → `3_seurat_clusters_tcga.r`).

**Winning parameters**
- `norm_method = LogNormalize`
- `selection.method = mean.var.plot` (no `nfeatures` cap)
- `dims = 1:30` PCs
- `k.param = 5` for `FindNeighbors`
- `resolution = 0.5` for `FindClusters`

**Inputs (replace paths with your own)**
- `target_seurat_all_genes.qs` — Seurat object with one cell per (slide, domain), counts = mean expression of the domain. Built upstream from the SpaGCN outputs.

**Outputs**
- `tcga_target_seurat_all_genes_new.qs` — Seurat object with `seurat_clusters` (0–10) and `mapped_clusters` (`Cluster_1` … `Cluster_11`)
- `TCGA_mapped_clusters.csv` — table mapping each refined per-spot domain to its `mapped_clusters` label

## Setup

In [ ]:
Sys.setenv(MKL_THREADING_LAYER = "GNU")
library(presto)
library(reticulate)
library(data.table)
library(Seurat)
library(Matrix)
library(qs)
library(dplyr)

# Replace with your local Python environment if you need reticulate for pd$read_pickle
use_python("/usr/local/Anaconda/envs/py3.10/bin/python3", required = TRUE)
pd <- import("pandas")

## Parameters (winning set)

In [ ]:
dims_end        <- 30
dims            <- 1:dims_end
k.param         <- 5
feat_sel_method <- "mean.var.plot"
norm_method     <- "LogNormalize"
res             <- 0.5

## Load the per-domain Seurat object

Each Seurat "cell" here is one SpaGCN domain on one slide; the count matrix entry is the mean expression of that gene across the domain's spots. Replace the path with your own copy of the file.

In [ ]:
target <- qs::qread("/vf/users/Ruppin_ST/scr/TCGA_SpaGCN/target_seurat_all_genes.qs")
target

## Seurat pipeline

Standard sequence: normalize → variable-feature selection → scale → PCA → SNN graph → Louvain clustering.

In [ ]:
target <- NormalizeData(target, normalization.method = norm_method)
target <- FindVariableFeatures(target, selection.method = feat_sel_method)
target <- ScaleData(target)
target <- RunPCA(target)
target <- FindNeighbors(target, dims = dims, k.param = k.param)
target <- FindClusters(
  target, resolution = res,
  cluster.name = paste0("clusters_", k.param, "_", length(dims), "_", res)
)

## Map cluster IDs to stable named labels (11 ST clusters)

Seurat returns numeric cluster IDs ordered by size; the paper relabels them to a fixed `Cluster_1`…`Cluster_11` naming so cluster identity is stable across reruns. The mapping below is from `wining_clustering_v2.r` — derived once and frozen.

In [ ]:
cluster_mapping <- c('0' = 'Cluster_1',
                     '1' = 'Cluster_2',
                     '10' = 'Cluster_3',
                     '2' = 'Cluster_4',
                     '3' = 'Cluster_5',
                     '4' = 'Cluster_6',
                     '5' = 'Cluster_7',
                     '6' = 'Cluster_8',
                     '7' = 'Cluster_9',
                     '8' = 'Cluster_10',
                     '9' = 'Cluster_11')

target@meta.data <- target@meta.data %>%
  mutate(mapped_clusters = recode(as.character(seurat_clusters), !!!cluster_mapping))

Idents(target) <- target@meta.data$mapped_clusters
table(Idents(target))

## Persist outputs

- A CSV mapping each refined per-spot domain ID to its cluster — used downstream by the SpatioType notebook (step 3) and by external cohorts (METABRIC, clinical) that transfer these labels via `Seurat::TransferData`.
- The Seurat object with `mapped_clusters` set as the default identity.

In [ ]:
gg <- target@meta.data[, c("mapped_clusters", "slide_name"), drop = FALSE]
gg$refined_pred <- rownames(gg)

fwrite(gg, file = "/vf/users/Ruppin_ST/scr/pipline/TCGA_mapped_clusters.csv")
qsave(target, file = "/vf/users/Ruppin_ST/scr/TCGA_SpaGCN/tcga_target_seurat_all_genes_new.qs")

## (Optional) Marker analysis

`FindAllMarkers` with MAST yields the per-cluster differentially expressed genes used in the paper's biological interpretation of each ST cluster. Runs on the full object or on cluster subsets.

In [ ]:
all_markers <- FindAllMarkers(target, only.pos = FALSE, test.use = "MAST")
fwrite(all_markers, file = "all_markers_MAST.csv")
head(all_markers)